# M3 — Wavelet detector (combined PTB-XL + Ningbo)

Time-frequency **LOCALIZATION** representation (energy dropped) + delineation-free **QRS-localized** families (QRS-mask, QRS-onset rising-edge, polarity). 100% event-free (no NeuroKit). SWT db4+sym4+coif3 + WPT db4 L6 + DWT db4.

Gabarit as M1/M2: cached extraction → FDR gate + cross-dataset + dedup → ablation → AP-vs-k → overfit diagnostic → 2D K×config grid → selection → `evaluate_standard` → freeze. XGBoost only, NaN-native, Platt on native OOF.

**Judge = pooled OOF AP (folds 1-8, stable); fold 9 = validation; fold 10 UNTOUCHED.**

### Selection note (differs from M1/M2 — on purpose)

M1/M2 select with the **1-SE rule** (smallest K within the bootstrap IC of the best depth-2 model under a train-gap cap). M3 differs, justified by this detector's regime:

- With 115 training WPW and a large localized pool, **`AP_train` saturates to ~1.0 for any depth ≥ 2** (block 7a.3c), so the train-gap (`AP_train − AP_oof`) is **not a valid overfit probe** here. The honest guards are the **pooled OOF AP** (stable, 115 WPW), the **inter-fold OOF AP variance**, and held-out **fold 9**.
- Selection = **max OOF AP, depth open (2/3/4)**, then a **parsimony tiebreak among true OOF-ties** (smallest K within `TIE_EPS`). The 1-SE rule over-pruned this pool.
- Depth is **data-driven** (same protocol as the other detectors); M3's larger pool legitimately supports more capacity. A follow-up re-checks M1/M2 under open depth for consistency. **Final unbiased performance = fold 10 (ensemble notebook).**

### Block 7a.0: Setup — paths, frozen filter, POOL switch, TIE_EPS, metadata

In [ ]:
# M3 — Wavelet detector, COMBINED (PTB-XL + Ningbo). Time-frequency LOCALIZATION representation
# (SWT db4+sym4+coif3 + WPT db4 L6 + DWT db4) + delineation-free QRS-localized families (QRS-mask,
# QRS-onset, polarity). 100% event-free (no NeuroKit). Judge: AP (pooled OOF folds 1-8 = stable; fold 9 =
# validation; fold 10 UNTOUCHED). Selection: max OOF AP, depth open (2/3/4), parsimony tiebreak among ties.
import os, sys, json
import numpy as np, pandas as pd
POOL = 'localization'   # 'localization' (final; energy dropped) | 'union' (diagnostic, keeps energy)
ROOT      = r'.'
PROCESSED = os.path.join(ROOT, 'data', 'processed')
MODELS    = os.path.join(ROOT, 'models', 'M3_wavelet')
FIG       = os.path.join(ROOT, 'reports', 'figures')
METRICS   = os.path.join(ROOT, 'reports', 'metrics')
SRC       = os.path.join(ROOT, 'src')
for d in (MODELS, FIG, METRICS): os.makedirs(d, exist_ok=True)
sys.path.insert(0, SRC)
from signal_loading import load_signal, LEADS_CANONICAL
from evaluation import evaluate_standard
print("signal_loading OK - canonical lead order:", LEADS_CANONICAL)
with open(os.path.join(PROCESSED, 'filter_config.json'), encoding='utf-8') as f:
    FCFG = json.load(f)['filter_FINAL']
assert (FCFG['low'], FCFG['high'], FCFG['order']) == (0.5, 40, 4), "Unexpected filter!"
FS = FCFG['fs']
print(f"Frozen filter: {FCFG['type']} order {FCFG['order']}, {FCFG['low']}-{FCFG['high']} Hz, {FCFG['phase']}")
LEADS_M3 = list(LEADS_CANONICAL); LEAD_IDX = {L: LEADS_CANONICAL.index(L) for L in LEADS_M3}
print(f"M3 leads: {len(LEADS_M3)} (all 12) | POOL = {POOL}")
TAG = 'combined'
FEATURES_CSV = os.path.join(PROCESSED, 'm3_features.csv')
TIE_EPS = 0.01   # parsimony tiebreak band on OOF AP (true ties -> smallest K)
meta = pd.read_csv(os.path.join(PROCESSED, 'metadata_combined.csv'), dtype={'ecg_id': str})
n_wpw = int((meta.label == 1).sum())
print(f"\nmetadata_combined: {len(meta)} ECGs, {n_wpw} WPW")
print(f"WPW per fold: {meta[meta.label==1].groupby('fold').size().to_dict()}")
print(f"Train(1-8): {int(meta[(meta.fold.between(1,8))&(meta.label==1)].shape[0])} | "
      f"Val(9): {int(meta[(meta.fold==9)&(meta.label==1)].shape[0])} | "
      f"Test(10): {int(meta[(meta.fold==10)&(meta.label==1)].shape[0])} [UNTOUCHED]")
assert n_wpw == 142, "Expected 142 WPW!"
print("\nBlock 7a.0 OK.")

### Block 7a.0b: Patient-leakage check across folds (blocking assert)

In [ ]:
# Patient-leakage check across folds (blocking assert).
pat_folds = meta.groupby('patient_id')['fold'].nunique(); leaky = pat_folds[pat_folds > 1]
print(f"Unique patients: {meta.patient_id.nunique()} | ECGs: {len(meta)} | in >1 fold: {len(leaky)}")
print(f"WPW: {int((meta.label==1).sum())} ECGs across {meta[meta.label==1].patient_id.nunique()} unique patients")
assert len(leaky) == 0, f"PATIENT LEAK: {len(leaky)} patients in multiple folds."
print("[OK] No patient leakage across folds.")

### Block 7a.1: Wavelet + QRS-localized feature extraction — cached, guarded, tqdm

In [ ]:
# Massive wavelet feature extraction (localization-oriented, 100% event-free) + delineation-free
# QRS-localized families. Transforms: SWT (db4+sym4+coif3) + DWT (db4) on A6,D6-D3 ; WPT (db4) L6.
# Families: (i) energy/entropy, (ii) localization/transient (+ Hjorth, temporal-asymmetry, cross-scale
# coherence). QRS-localized (db4 only): a QRS locator = smoothed |D3|+|D4| envelope (mid-band sharpness);
# low-freq A6/D6/D5 weighted by it -> energy at QRS (qrswe), enrichment (qrsfrac), peak (qrsmax),
# low/sharp ratio (qrs2mid); onset (rising-edge) variants (qrsonwe/qrsonfrac/qrson2mid); polarity
# (signed onset amplitude qrspolsigned + consistency qrspolcons). Robust NaN/inf. Anti-rerun guard.
FORCE_REEXTRACT = False
RUN_FULL        = True

import warnings, time, contextlib
import numpy as np
import pywt
from scipy.signal import butter, sosfiltfilt
from scipy.stats import skew, kurtosis
from joblib import Parallel, delayed
from tqdm import tqdm
warnings.filterwarnings('ignore')

SOS_BP = butter(FCFG['order'], [FCFG['low']/(FS/2), FCFG['high']/(FS/2)], btype='band', output='sos')
def bp(x): return sosfiltfilt(SOS_BP, np.asarray(x, dtype=np.float64))

@contextlib.contextmanager
def tqdm_joblib(t):
    import joblib
    class _Cb(joblib.parallel.BatchCompletionCallBack):
        def __call__(self,*a,**k): t.update(n=self.batch_size); return super().__call__(*a,**k)
    old=joblib.parallel.BatchCompletionCallBack; joblib.parallel.BatchCompletionCallBack=_Cb
    try: yield t
    finally: joblib.parallel.BatchCompletionCallBack=old; t.close()

EPS=1e-12
SWT_LEVEL=6; WPT_LEVEL=6; PAD_LEN=5120
SWT_BANDS=['A6','D6','D5','D4','D3']
SWT_MOTHERS=[('db4','db4'),('sym4','sym4'),('coif3','coif3')]
QRS_WIN=max(3,int(0.04*FS)); QRS_KER=np.ones(QRS_WIN)/QRS_WIN

def _pad(x):
    n=len(x)
    return x[:PAD_LEN] if n>=PAD_LEN else np.pad(x,(0,PAD_LEN-n),mode='reflect')
def _gini(w):
    a=np.sort(np.abs(w)); n=a.size; s=a.sum()+EPS
    return float((2*np.arange(1,n+1)-n-1).dot(a)/(n*s))
def f_energy(c,p):
    e=float(np.sum(c**2)); return {f'{p}_energy':e, f'{p}_logenergy':float(np.log(e+EPS))}
def f_loc(c,p):
    a=np.abs(c); rms=float(np.sqrt(np.mean(c**2)))+EPS; sd=float(np.std(c))
    d1=np.diff(c); v0=np.var(c)+EPS; v1=np.var(d1)+EPS; d2=np.diff(d1); v2=np.var(d2)+EPS
    mob=np.sqrt(v1/v0); h=len(c)//2
    return {f'{p}_maxabs':float(a.max()), f'{p}_p95':float(np.percentile(a,95)),
            f'{p}_p99':float(np.percentile(a,99)), f'{p}_crest':float(a.max()/rms),
            f'{p}_std':sd, f'{p}_mad':float(np.median(np.abs(c-np.median(c)))),
            f'{p}_skew':float(skew(c)) if c.size>3 else np.nan,
            f'{p}_kurt':float(kurtosis(c)) if c.size>3 else np.nan,
            f'{p}_gini':_gini(c**2), f'{p}_nover3':float(np.mean(a>3*sd)) if sd>EPS else 0.0,
            f'{p}_hjmob':float(mob), f'{p}_hjcomp':float(np.sqrt(v2/v1)/(mob+EPS)),
            f'{p}_halfratio':float(np.sum(c[:h]**2)/(np.sum(c**2)+EPS))}
def swt_features(x,wavelet,wname,L):
    out={}
    try: coeffs=pywt.swt(x,wavelet,level=SWT_LEVEL,trim_approx=True,norm=True)
    except Exception: return out
    bands={SWT_BANDS[i]:coeffs[i] for i in range(len(SWT_BANDS))}
    tot=sum(float(np.sum(c**2)) for c in bands.values())+EPS
    for bn,c in bands.items():
        p=f'swt{wname}_{bn}_{L}'; out.update(f_energy(c,p)); out.update(f_loc(c,p)); out[f'{p}_relenergy']=float(np.sum(c**2)/tot)
    pe=np.array([np.sum(bands[b]**2) for b in bands]); pe=pe/(pe.sum()+EPS)
    out[f'swt{wname}_wentropy_{L}']=float(-np.sum(pe*np.log(pe+EPS)))
    for b1,b2 in [('D6','D5'),('D5','D4'),('D4','D3'),('D6','D4'),('A6','D6')]:
        c1,c2=bands[b1],bands[b2]
        out[f'swt{wname}_coh{b1}{b2}_{L}']=float(np.corrcoef(c1,c2)[0,1]) if np.std(c1)>EPS and np.std(c2)>EPS else np.nan
    if wname=='db4':
        mid=np.abs(bands['D3'])+np.abs(bands['D4'])
        env=np.convolve(mid,QRS_KER,mode='same'); mw=env/(env.sum()+EPS)
        hi=env>=np.percentile(env,90); mid_we=float(np.sum(mid**2*mw))
        denv=np.diff(env,prepend=env[0]); onset=env*np.maximum(denv,0.0); ow=onset/(onset.sum()+EPS)
        mid_we_on=float(np.sum(mid**2*ow))
        for bn in ('A6','D6','D5'):
            c=bands[bn]; p=f'swt{wname}_{bn}_{L}'; we=float(np.sum(c**2*mw)); won=float(np.sum(c**2*ow))
            out[f'{p}_qrswe']=we; out[f'{p}_qrsfrac']=we/(float(np.mean(c**2))+EPS)
            out[f'{p}_qrsmax']=float(np.max(np.abs(c)[hi])) if hi.any() else 0.0
            out[f'{p}_qrs2mid']=we/(mid_we+EPS)
            out[f'{p}_qrsonwe']=won; out[f'{p}_qrsonfrac']=won/(float(np.mean(c**2))+EPS)
            out[f'{p}_qrson2mid']=won/(mid_we_on+EPS)
            sgn=float(np.sum(c*ow)); out[f'{p}_qrspolsigned']=sgn
            out[f'{p}_qrspolcons']=abs(sgn)/(float(np.sum(np.abs(c)*ow))+EPS)
    return out
def dwt_features(x,wavelet,wname,L):
    out={}
    try: coeffs=pywt.wavedec(x,wavelet,level=SWT_LEVEL,mode='periodization')
    except Exception: return out
    bands={SWT_BANDS[i]:coeffs[i] for i in range(len(SWT_BANDS))}
    tot=sum(float(np.sum(c**2)) for c in bands.values())+EPS
    for bn,c in bands.items():
        p=f'dwt{wname}_{bn}_{L}'; out.update(f_energy(c,p)); out.update(f_loc(c,p)); out[f'{p}_relenergy']=float(np.sum(c**2)/tot)
    return out
def wpt_features(x,wavelet,L):
    out={}
    try:
        wp=pywt.WaveletPacket(x,wavelet,mode='periodization',maxlevel=WPT_LEVEL)
        nodes=wp.get_level(WPT_LEVEL,order='freq')[:13]; datas=[np.asarray(n.data) for n in nodes]
    except Exception: return out
    tot=sum(float(np.sum(d**2)) for d in datas)+EPS
    for i,d in enumerate(datas):
        p=f'wptdb4_b{i}_{L}'
        out[f'{p}_energy']=float(np.sum(d**2)); out[f'{p}_relenergy']=float(np.sum(d**2)/tot)
        out[f'{p}_maxabs']=float(np.max(np.abs(d))); out[f'{p}_crest']=float(np.max(np.abs(d))/(np.sqrt(np.mean(d**2))+EPS))
    return out
def extract_lead(xf,L):
    xp=_pad(xf); o={}
    for wv,wn in SWT_MOTHERS: o.update(swt_features(xp,wv,wn,L))
    o.update(dwt_features(xp,'db4','db4',L)); o.update(wpt_features(xp,'db4',L))
    return o
def process_one(m):
    warnings.filterwarnings('ignore')
    row={'ecg_id':m['ecg_id'],'patient_id':m['patient_id'],'label':m['label'],'fold':m['fold'],'source':m['source'],'extraction_failed':0}
    try:
        sig=load_signal(m['ecg_id'],m['source'])
        for L in LEADS_M3: row.update(extract_lead(bp(sig[:,LEAD_IDX[L]]),L))
    except Exception: row['extraction_failed']=1
    return row

print("Synthetic test...")
t=np.arange(5000)/FS; synth=(np.sin(2*np.pi*1.2*t)+0.3*np.random.randn(5000))
fe=extract_lead(bp(synth),'II'); nfeat=len(fe)
print(f"  OK - {nfeat} features/lead (qrs-localized {sum('qrs' in k for k in fe)}), ~{nfeat*len(LEADS_M3)} features/ECG.")
assert all(np.isfinite(v) or (isinstance(v,float) and np.isnan(v)) for v in fe.values()), "non-numeric value!"
if os.path.exists(FEATURES_CSV) and not FORCE_REEXTRACT:
    print(f"\n{FEATURES_CSV} exists -> extraction SKIPPED.")
else:
    recs=meta.to_dict('records'); sample=recs[:200]; t0=time.time()
    with tqdm_joblib(tqdm(total=len(sample),desc='Timing 200',unit='ecg')):
        _=Parallel(n_jobs=10,backend='loky')(delayed(process_one)(m) for m in sample)
    eta=(time.time()-t0)/len(sample)*len(recs)/60
    print(f"\n  200 ECGs in {time.time()-t0:.1f}s -> full-run ETA ~ {eta:.0f} min ({eta/60:.1f} h).")
    if RUN_FULL:
        t0=time.time()
        with tqdm_joblib(tqdm(total=len(recs),desc='M3 wavelet extraction',unit='ecg')):
            rows=Parallel(n_jobs=10,backend='loky')(delayed(process_one)(m) for m in recs)
        df_=pd.DataFrame(rows); df_.to_csv(FEATURES_CSV,index=False)
        mc=['ecg_id','patient_id','label','fold','source','extraction_failed']
        print(f"\nDone in {(time.time()-t0)/60:.1f} min -> {FEATURES_CSV} | shape {df_.shape} ({df_.shape[1]-len(mc)} feats)")
        for s in df_.source.unique():
            sub=df_[df_.source==s]; print(f"  {s}: failed WPW {sub[sub.label==1].extraction_failed.mean()*100:.1f}% | non-WPW {sub[sub.label==0].extraction_failed.mean()*100:.1f}%")

### Block 7a.2: Feature selection (FDR gate + cross-dataset) + localization pool + dedup>0.95

In [ ]:
# Feature selection. `passed` = full gate survivors. FINAL pool = localization (energy/entropy dropped;
# ablation showed energy AP ~0.08, dead weight). Includes the QRS-localized families (non-energy). dedup>0.95.
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm
df = pd.read_csv(FEATURES_CSV, dtype={'ecg_id': str})
META = ['ecg_id','patient_id','label','fold','source','extraction_failed']
ALL_FEATS = [c for c in df.columns if c not in META]
tr = df[df.fold.between(1,8)]
print(f"Candidate features: {len(ALL_FEATS)} (qrs-localized {sum('qrs' in c for c in ALL_FEATS)}) | "
      f"train 1-8: {len(tr)} ECGs, {int((tr.label==1).sum())} WPW")
def cohens_d(a,b):
    a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return np.nan
    sp=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
    return (a.mean()-b.mean())/sp if sp>0 else np.nan
def d_ci(a,b,n=1000,seed=42):
    rng=np.random.default_rng(seed); a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return (np.nan,np.nan)
    ds=[cohens_d(rng.choice(a,len(a),True),rng.choice(b,len(b),True)) for _ in range(n)]
    return tuple(np.nanpercentile(ds,[2.5,97.5]))
SEL_CSV = os.path.join(PROCESSED,'m3_combined_selection.csv')
if os.path.exists(SEL_CSV) and set(pd.read_csv(SEL_CSV, usecols=['feature'])['feature']) == set(ALL_FEATS):
    res = pd.read_csv(SEL_CSV); print(f"[guard] {os.path.basename(SEL_CSV)} reloaded ({len(res)} feats).")
else:
    w,n=tr[tr.label==1],tr[tr.label==0]; ptb,nin=tr[tr.source=='ptbxl'],tr[tr.source=='ningbo']; rows=[]
    for c in tqdm(ALL_FEATS, desc='Gate (d+p+IC+cross)', unit='feat'):
        a,b=w[c].values.astype(float),n[c].values.astype(float); d=cohens_d(a,b)
        try: _,p=mannwhitneyu(a[~np.isnan(a)],b[~np.isnan(b)],alternative='two-sided')
        except Exception: p=np.nan
        lo,hi=d_ci(a,b)
        dp=cohens_d(ptb[ptb.label==1][c].values.astype(float),ptb[ptb.label==0][c].values.astype(float))
        dn=cohens_d(nin[nin.label==1][c].values.astype(float),nin[nin.label==0][c].values.astype(float))
        cross_ok=(np.isfinite(dp) and np.isfinite(dn) and np.sign(dp)==np.sign(dn) and abs(dp)>0.2 and abs(dn)>0.2)
        ci_ok=(np.isfinite(lo) and np.isfinite(hi) and (lo>0)==(hi>0))
        rows.append({'feature':c,'d':d,'d_ptb':dp,'d_nin':dn,'p_raw':p,'ci_excl0':ci_ok,'cross_ok':cross_ok})
    res=pd.DataFrame(rows); ok=res.p_raw.notna()
    res.loc[ok,'p_FDR']=multipletests(res.loc[ok,'p_raw'],method='fdr_bh')[1]
    res['gate']=(res.d.abs()>0.3)&(res.p_FDR<0.05)&res.ci_excl0&res.cross_ok
    res=res.sort_values('d',key=lambda s:s.abs(),ascending=False).reset_index(drop=True)
    res.to_csv(SEL_CSV,index=False)
passed=res[res.gate].feature.tolist()
def _isenergy(c): return ('energy' in c) or ('wentropy' in c)
pool_feats = [c for c in passed if not _isenergy(c)] if POOL=='localization' else list(passed)
print(f"Gate survivors: {len(passed)} | POOL='{POOL}' -> pool: {len(pool_feats)} (qrs-localized {sum('qrs' in c for c in pool_feats)})")
def dedup_fast(feats, thr=0.95):
    sub=tr[feats].rank().to_numpy(); cm=np.nanmean(sub,axis=0)
    ii=np.where(np.isnan(sub)); sub[ii]=np.take(cm,ii[1])
    C=np.abs(np.corrcoef(sub,rowvar=False)); idx={f:i for i,f in enumerate(feats)}; keep=[]
    for f in tqdm(feats, desc=f'dedup>{thr}', unit='feat'):
        if all(C[idx[f],idx[k]]<=thr for k in keep): keep.append(f)
    return keep
dedup_list = dedup_fast(pool_feats,0.95) if pool_feats else []
print(f"After dedup>0.95 ({POOL} pool): {len(dedup_list)} features (qrs-localized {sum('qrs' in c for c in dedup_list)})\n")
def _transform(c): return ('SWT' if c.startswith('swt') else 'DWT' if c.startswith('dwt') else 'WPT' if c.startswith('wpt') else '?')
def _mother(c):    return ('db4' if 'db4' in c else 'sym4' if 'sym4' in c else 'coif3' if 'coif3' in c else '-')
rep=pd.Series(dedup_list)
print("Final-pool by transform:", rep.map(_transform).value_counts().to_dict())
print("Final-pool by mother   :", rep.map(_mother).value_counts().to_dict())
pd.set_option('display.float_format',lambda x:f'{x:.3f}')
print("\nTop 15 in final pool:")
print(res[res.feature.isin(dedup_list)][['feature','d','d_ptb','d_nin','p_FDR']].head(15).to_string(index=False))
print("\nTop 6 onset / Top 6 polarity features (by |d|, in pool):")
for tagn in ('qrson','qrspol'):
    sub=res[res.feature.isin(dedup_list) & res.feature.str.contains(tagn)]
    print(f"  [{tagn}]"); print(sub[['feature','d','d_ptb','d_nin']].head(6).to_string(index=False) if len(sub) else "    (none survived)")

### Block 7a.2b: Histograms of top selected features

In [ ]:
# Histograms of the top-6 selected features (WPW vs non-WPW, folds 1-8).
import matplotlib.pyplot as plt
top6 = dedup_list[:6]
fig,ax=plt.subplots(2,3,figsize=(15,7))
for j,f in enumerate(top6):
    a=ax[j//3,j%3]; wv=tr[tr.label==1][f].dropna(); nv=tr[tr.label==0][f].dropna()
    lo,hi=np.nanpercentile(tr[f],1),np.nanpercentile(tr[f],99); bins=np.linspace(lo,hi,40) if hi>lo else 40
    a.hist(nv,bins=bins,density=True,alpha=.5,color='#94a3b8',label='non-WPW')
    a.hist(wv,bins=bins,density=True,alpha=.6,color='#dc2626',label='WPW')
    a.set_title(f"{f}\n(d={res.set_index('feature').loc[f,'d']:.2f})",fontsize=8); a.legend(fontsize=7)
plt.suptitle('M3 combined - top-6 selected wavelet features (folds 1-8)'); plt.tight_layout()
plt.savefig(os.path.join(FIG,'m3_combined_histograms.png'),dpi=130,bbox_inches='tight'); plt.show()

### Block 7a.2c: QRS sub-family ablation (localization / +qrs-mask / +onset+polarity) + helpers

In [ ]:
# Modeling helpers + QRS sub-family ablation: localization vs +qrs-mask vs +onset+polarity. Quantifies the
# incremental value of each delineation-free QRS-localized family. Coarse OOF AP-vs-k, cached.
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier
from tqdm import tqdm
import matplotlib.pyplot as plt
d8 = df[df.fold.between(1,8)].reset_index(drop=True); y = d8.label.values; folds = d8.fold.values
FOLD_MASKS = [(folds!=h, folds==h) for h in sorted(np.unique(folds))]
f9 = df[df.fold==9].reset_index(drop=True); yf9 = f9.label.values
spw8 = (y==0).sum()/max((y==1).sum(),1)
def make_xgb(spw=None,**kw):
    if spw is None: spw=spw8
    p=dict(n_estimators=100,max_depth=2,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,
           reg_lambda=2.0,min_child_weight=3,scale_pos_weight=spw,eval_metric='aucpr',
           tree_method='hist',random_state=42,n_jobs=10); p.update(kw); return XGBClassifier(**p)
def oof_xgb(feats,**kw):
    X=d8[feats]; oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        spw=(y[trm]==0).sum()/max((y[trm]==1).sum(),1)
        oof[vam]=make_xgb(spw,**kw).fit(X[trm],y[trm]).predict_proba(X[vam])[:,1]
    return oof
def oof_ap(feats,**kw):
    oof=oof_xgb(feats,**kw); m=~np.isnan(oof); return float(average_precision_score(y[m],oof[m]))
def ap_boot(yy,pp,n=1000,seed=42):
    rng=np.random.default_rng(seed); pos,neg=np.where(yy==1)[0],np.where(yy==0)[0]; a=np.empty(n)
    for i in range(n):
        idx=np.concatenate([rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)]); a[i]=average_precision_score(yy[idx],pp[idx])
    return np.median(a),np.percentile(a,2.5),np.percentile(a,97.5)
def dedup_within(feats,thr=0.95):
    if not feats: return []
    sub=tr[feats].rank().to_numpy(); cm=np.nanmean(sub,axis=0)
    ii=np.where(np.isnan(sub)); sub[ii]=np.take(cm,ii[1])
    C=np.abs(np.corrcoef(sub,rowvar=False)); idx={f:i for i,f in enumerate(feats)}; keep=[]
    for f in feats:
        if all(C[idx[f],idx[k]]<=thr for k in keep): keep.append(f)
    return keep
_isnew=lambda c: ('qrson' in c) or ('qrspol' in c)
P_loc = dedup_within([c for c in pool_feats if 'qrs' not in c],0.95)
P_qrs = dedup_within([c for c in pool_feats if not _isnew(c)],0.95)   # localization + qrs-mask (no onset/pol)
P_all = dedup_list                                                    # + onset + polarity
print(f"ablation pools: localization={len(P_loc)} | +qrs_mask={len(P_qrs)} | +onset_polarity={len(P_all)}")
QRS_ABL_CSV=os.path.join(PROCESSED,'m3_combined_qrsablation.csv')
if os.path.exists(QRS_ABL_CSV):
    qc=pd.read_csv(QRS_ABL_CSV); print(f"[guard] {os.path.basename(QRS_ABL_CSV)} reloaded.")
else:
    rowsA=[]
    for name,dl in [('localization',P_loc),('+qrs_mask',P_qrs),('+onset_polarity',P_all)]:
        ks=list(range(20,len(dl)+1,40))+[len(dl)]
        for k in tqdm(ks,desc=f'ablation {name}',unit='k'):
            rowsA.append({'pool':name,'k':k,'AP':oof_ap(dl[:k]),'n':len(dl)})
    qc=pd.DataFrame(rowsA); qc.to_csv(QRS_ABL_CSV,index=False)
plt.figure(figsize=(11,5)); qrs_abl={}
for name in ('localization','+qrs_mask','+onset_polarity'):
    cc=qc[qc['pool']==name]
    if len(cc)==0: continue
    kb=int(cc.loc[cc.AP.idxmax(),'k']); ab=float(cc.AP.max()); qrs_abl[name]=dict(k=kb,ap=ab)
    plt.plot(cc.k,cc.AP,'o-',ms=3,label=f'{name} (best {ab:.3f} @ k={kb})')
    print(f"  {name:16s}: best OOF AP {ab:.3f} @ k={kb}")
plt.axhline(0.299,ls='--',color='#7c3aed',label='M2 OOF (0.299)'); plt.axhline(0.198,ls=':',color='#16a34a',label='M1 OOF (0.198)')
plt.xlabel('k'); plt.ylabel('AP OOF folds 1-8'); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.title('M3 combined - QRS sub-family ablation (localization / +qrs-mask / +onset+polarity)'); plt.tight_layout()
plt.savefig(os.path.join(FIG,'m3_combined_qrs_ablation.png'),dpi=140,bbox_inches='tight'); plt.show()
if all(k in qrs_abl for k in ('localization','+qrs_mask','+onset_polarity')):
    l1=qrs_abl['+qrs_mask']['ap']-qrs_abl['localization']['ap']; l2=qrs_abl['+onset_polarity']['ap']-qrs_abl['+qrs_mask']['ap']
    print(f"\n=== ABLATION === qrs-mask lift {l1:+.3f} | onset+polarity lift {l2:+.3f} -> "
          f"{'onset+polarity helps' if l2>0.01 else 'onset+polarity no clear gain'}")

### Block 7a.3: AP-vs-k + bootstrap 95% CI band — K_auto (OOF scale)

In [ ]:
# Production AP-vs-k on the final pool. FINE point curve (step 2) + bootstrap 95% CI band on a coarse
# k-subset (uncertainty on the curve) + tqdm + cached. K_auto = smallest k within 95% of the max OOF AP.
import matplotlib.pyplot as plt
RK_CSV = os.path.join(PROCESSED, 'm3_combined_ap_vs_k.csv')
CI_CSV = os.path.join(PROCESSED, 'm3_combined_ap_vs_k_ci.csv')
if os.path.exists(RK_CSV):
    rk=pd.read_csv(RK_CSV); print(f"[guard] {os.path.basename(RK_CSV)} reloaded ({len(rk)} k).")
else:
    ks=list(range(1,len(dedup_list)+1,2)); rows=[]
    for k in tqdm(ks, desc='AP vs k (step 2)', unit='k'):
        rows.append({'k':k,'AP_oof':oof_ap(dedup_list[:k])})
        if k%21==0: pd.DataFrame(rows).to_csv(RK_CSV,index=False)
    rk=pd.DataFrame(rows); rk.to_csv(RK_CSV,index=False)
# bootstrap 95% CI band on a coarse k-subset (cheap: 200 resamples, ~12-15 points)
if os.path.exists(CI_CSV):
    rkci=pd.read_csv(CI_CSV); print(f"[guard] {os.path.basename(CI_CSV)} reloaded ({len(rkci)} k).")
else:
    ks_ci=sorted(set(list(range(20,len(dedup_list)+1,40))+[len(dedup_list)])); rows=[]
    for k in tqdm(ks_ci, desc='AP-vs-k bootstrap CI', unit='k'):
        oof=oof_xgb(dedup_list[:k]); m=~np.isnan(oof); med,lo,hi=ap_boot(y[m],oof[m],n=200)
        rows.append({'k':k,'AP':med,'lo':lo,'hi':hi})
    rkci=pd.DataFrame(rows); rkci.to_csv(CI_CSV,index=False)
plt.figure(figsize=(11,5))
plt.fill_between(rkci.k,rkci.lo,rkci.hi,color='#2563eb',alpha=.15,label='95% bootstrap CI')
plt.plot(rk.k,rk.AP_oof,'-',color='#2563eb',lw=1,label='AP OOF folds 1-8 (stable juge)')
plt.axhline(0.198,ls=':',color='#16a34a',label='M1 OOF (0.198)'); plt.axhline(0.299,ls='--',color='#7c3aed',label='M2 OOF (0.299)')
plt.xlabel('k'); plt.ylabel('AP OOF'); plt.legend(); plt.grid(alpha=.3); plt.title('M3 combined - AP vs k [OOF scale, all detectors comparable]')
plt.savefig(os.path.join(FIG,'m3_combined_ap_vs_k.png'),dpi=150,bbox_inches='tight'); plt.show()
kbest=rk.loc[rk.AP_oof.idxmax()]; K_auto=int(rk[rk.AP_oof>=0.95*kbest.AP_oof].k.min())
print(f"Max OOF AP at k={int(kbest.k)} ({kbest.AP_oof:.4f}); 95% plateau from k={K_auto}.")
print(f"k_max ({POOL} dedup) = {len(dedup_list)} | provisional K_auto = {K_auto}")

### Block 7a.3c: Overfit diagnostic — AP_train saturation (depth 2/3/4); defines diag_one

In [ ]:
# Overfit diagnostic. Defines diag_one. Shows AP_train saturates (~1.0) with scarce positives (115 WPW)
# -> the train-gap is an invalid overfit probe here; we judge on OOF + fold9 + inter-fold variance.
# Depth 2/3/4 at K_auto. Cached.
GRIDC_CSV = os.path.join(PROCESSED,'m3_combined_hpgrid_diag.csv')
y8, y9 = d8.label.values, f9.label.values
def diag_one(cfg, feats):
    X8=d8[feats]; X9=f9[feats]; oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y8[trm].sum()==0 or y8[vam].sum()==0: continue
        spw=(y8[trm]==0).sum()/max((y8[trm]==1).sum(),1)
        oof[vam]=make_xgb(spw,**cfg).fit(X8[trm],y8[trm]).predict_proba(X8[vam])[:,1]
    m=~np.isnan(oof); ao=average_precision_score(y8[m],oof[m])
    mdl=make_xgb(**cfg).fit(X8,y8); at=average_precision_score(y8,mdl.predict_proba(X8)[:,1])
    af=average_precision_score(y9,mdl.predict_proba(X9)[:,1]) if y9.sum()>0 else np.nan
    return ao,at,at-ao,af
K_DIAG=K_auto; FEATS_DIAG=dedup_list[:K_DIAG]
if os.path.exists(GRIDC_CSV):
    G=pd.read_csv(GRIDC_CSV); print(f"[guard] {os.path.basename(GRIDC_CSV)} reloaded.")
else:
    GRID=[dict(max_depth=d,learning_rate=lr,n_estimators=200,min_child_weight=3) for d in (2,3,4) for lr in (0.05,0.1)]
    rows=[]
    for g in tqdm(GRID, desc='diag depth 2/3/4', unit='cfg'):
        ao,at,gp,af=diag_one(g,FEATS_DIAG); rows.append({**g,'AP_oof':ao,'AP_train':at,'gap':gp,'AP_f9':af})
    G=pd.DataFrame(rows); G.to_csv(GRIDC_CSV,index=False)
pd.set_option('display.float_format',lambda x:f'{x:.4f}')
print(f"At K={K_DIAG}:"); print(G[['max_depth','learning_rate','AP_oof','AP_train','gap','AP_f9']].to_string(index=False))
print(f"\nAP_train median = {G.AP_train.median():.3f} -> saturated. Judge on OOF + fold9 + inter-fold variance.")

### Block 7a.3d: 2D grid depth OPEN (2/3/4) + max-OOF selection + parsimony tiebreak + guards

In [ ]:
# 2D K x config grid, depth OPEN (2/3/4) + selection = MAX OOF AP with a PARSIMONY TIEBREAK among true
# OOF-ties: eligible = configs within TIE_EPS of the max OOF AP; among them take the SMALLEST K (then best
# fold9). Parsimony among genuine ties only (NOT 1-SE over-pruning). Guards: inter-fold OOF AP variance +
# fold9. NOTE: M1/M2 use a 1-SE rule; M3 differs on purpose (see the markdown note above). Final test = fold10.
GRID2D_CSV = os.path.join(PROCESSED,'m3_combined_grid2d.csv')
kmax=len(dedup_list)
KS = sorted(set([k for k in [40,80,140,220,320,420,500] if k<kmax] + [K_auto, kmax]))
CFGS={}
for dep,lrs in [(2,(0.03,0.05)),(3,(0.03,0.05,0.1)),(4,(0.03,0.05,0.1))]:
    for lr in lrs:
        CFGS[f'd{dep}_lr{int(lr*100):02d}']=dict(max_depth=dep,learning_rate=lr,n_estimators=200,min_child_weight=3)
print(f"KS grid: {KS} | configs: {list(CFGS)}")
if os.path.exists(GRID2D_CSV):
    G2=pd.read_csv(GRID2D_CSV); print(f"[guard] {os.path.basename(GRID2D_CSV)} reloaded.")
else:
    rows=[]
    for k in tqdm(KS, desc='K x config (depth 2/3/4)', unit='K'):
        for cn,c in CFGS.items():
            ao,at,gp,af=diag_one(c,dedup_list[:k]); rows.append({'K':k,'cfg':cn,'depth':c['max_depth'],'AP_oof':ao,'AP_train':at,'gap':gp,'AP_f9':af})
    G2=pd.DataFrame(rows); G2.to_csv(GRID2D_CSV,index=False)
pd.set_option('display.float_format',lambda x:f'{x:.4f}')
print(G2.sort_values('AP_oof',ascending=False).head(15).to_string(index=False))
def oof_perfold_ap(feats,**kw):
    oof=oof_xgb(feats,**kw); aps=[]
    for h in sorted(np.unique(folds)):
        msk=(folds==h)&(~np.isnan(oof))
        if y[msk].sum()>0: aps.append(average_precision_score(y[msk],oof[msk]))
    return np.array(aps)
maxoof=float(G2.AP_oof.max()); elig=G2[G2.AP_oof>=maxoof-TIE_EPS].copy()
print(f"\nMax OOF AP = {maxoof:.3f}; eligible (within {TIE_EPS}) = {len(elig)} configs:")
print(elig.sort_values(['K','AP_f9'],ascending=[True,False])[['K','cfg','depth','AP_oof','AP_f9']].to_string(index=False))
best=elig.sort_values(['K','AP_f9'],ascending=[True,False]).iloc[0]   # parsimony: smallest K, then best fold9
K=int(best.K); CFGNAME=best['cfg']; FEATURES_K=dedup_list[:K]
CFG={**CFGS[CFGNAME],'subsample':0.8,'colsample_bytree':0.8,'reg_lambda':2.0}
pf=oof_perfold_ap(FEATURES_K,**CFGS[CFGNAME])
fragile=(pf.std()>0.15) or (pf.min()<0.10) or (best.AP_f9<0.5*best.AP_oof)
print(f"\n[selection] MAX OOF AP, depth open, parsimony tiebreak among true ties (TIE_EPS={TIE_EPS}).")
print(f"  >>> FINAL: K={K}, cfg={CFGNAME} (depth {int(best.depth)}) | OOF AP={best.AP_oof:.3f} | fold9={best.AP_f9:.3f} | gap(train,ignored)={best.gap:.3f}")
print(f"  guards: per-fold OOF AP {pf.mean():.3f}+/-{pf.std():.3f} (range {pf.min():.3f}-{pf.max():.3f}); fold9 corroborates={'yes' if best.AP_f9>=best.AP_oof-0.10 else 'weak'}")
if fragile: print("  [WARNING] winner looks fragile (high inter-fold variance / weak fold9) - inspect; fold10 (ensemble notebook) is the final arbiter.")

### Block 7a.4: Final model + standardized eval + multi-seed + screening metrics

In [ ]:
# Final M3 model + standardized evaluation + multi-seed + screening metrics (precision@recall>=0.8 and a
# recall-oriented operating point: threshold chosen on OOF for recall>=0.90, applied to fold9).
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression as _LR
from tqdm import tqdm
y8,y9=d8.label.values,f9.label.values; X8,X9=d8[FEATURES_K],f9[FEATURES_K]
def make_final(**kw):
    p=dict(**CFG,scale_pos_weight=spw8,eval_metric='aucpr',tree_method='hist',random_state=42,n_jobs=10); p.update(kw); return XGBClassifier(**p)
xgb_raw=make_final().fit(X8,y8); score9=xgb_raw.predict_proba(X9)[:,1]
ap_train=average_precision_score(y8,xgb_raw.predict_proba(X8)[:,1])
oof_raw=np.full(len(d8),np.nan)
for trm,vam in FOLD_MASKS:
    if y8[trm].sum()==0 or y8[vam].sum()==0: continue
    spw=(y8[trm]==0).sum()/max((y8[trm]==1).sum(),1)
    oof_raw[vam]=make_final(scale_pos_weight=spw).fit(X8[trm],y8[trm]).predict_proba(X8[vam])[:,1]
mask=~np.isnan(oof_raw)
platt=_LR(max_iter=2000).fit(oof_raw[mask].reshape(-1,1),y8[mask]); score9_cal=platt.predict_proba(score9.reshape(-1,1))[:,1]
aps,aucs=[],[]
for sd in tqdm(range(10), desc='multi-seed', unit='seed'):
    mmd=make_final(random_state=sd).fit(X8,y8); pp=mmd.predict_proba(X9)[:,1]
    aps.append(average_precision_score(y9,pp)); aucs.append(roc_auc_score(y9,pp))
MULTISEED=dict(AP_mean=float(np.mean(aps)),AP_std=float(np.std(aps)),AUC_mean=float(np.mean(aucs)),AUC_std=float(np.std(aucs)))
print(f"Multi-seed fold9: AP {np.mean(aps):.3f}+/-{np.std(aps):.3f} | AUC {np.mean(aucs):.3f}+/-{np.std(aucs):.3f}")
ap_oof_final=average_precision_score(y8[mask],oof_raw[mask])
print(f"OOF AP folds 1-8 (stable juge) = {ap_oof_final:.3f}  [vs M1 OOF 0.198, M2 OOF 0.299]")
def prec_at_recall(yt,sc,target=0.8):
    order=np.argsort(-np.asarray(sc)); yy=np.asarray(yt)[order]
    tp=np.cumsum(yy); P=yy.sum(); rec=tp/P; prec=tp/np.arange(1,len(yy)+1)
    ok=np.where(rec>=target)[0]
    return (float(prec[ok[0]]),float(rec[ok[0]])) if len(ok) else (float('nan'),float('nan'))
p80,r80=prec_at_recall(y9,score9,0.8); print(f"Screening: precision@recall>=0.8 (fold9) = {p80:.3f} at recall {r80:.3f}")
def thr_for_recall(yo,so,target=0.90):
    order=np.argsort(-np.asarray(so)); ys=np.asarray(yo)[order]; ss=np.asarray(so)[order]
    tp=np.cumsum(ys); P=ys.sum(); rec=tp/P; idx=np.where(rec>=target)[0]
    return float(ss[idx[0]]) if len(idx) else float(ss[-1])
thr90=thr_for_recall(y8[mask],oof_raw[mask],0.90); pred=(score9>=thr90).astype(int)
tp=int(((pred==1)&(y9==1)).sum()); fp=int(((pred==1)&(y9==0)).sum()); fn=int(((pred==0)&(y9==1)).sum()); tn=int(((pred==0)&(y9==0)).sum())
print(f"Recall-oriented op-point (thr@OOF recall>=0.90 = {thr90:.3f}) on fold9: TP {tp} FP {fp} FN {fn} TN {tn} | "
      f"recall {tp/(tp+fn) if (tp+fn) else 0:.3f} precision {tp/(tp+fp) if (tp+fp) else 0:.3f}")
M3_STD=evaluate_standard(f'M3_{TAG}',y_oof=y8[mask],score_oof=oof_raw[mask],y_test=y9,score_test=score9,
    figures_dir=FIG,metrics_dir=METRICS,score_test_calibrated=score9_cal,ap_train=ap_train,multiseed=MULTISEED,
    extra={'K':K,'POOL':POOL,'cfg':CFGNAME,'depth':int(best.depth),'OOF_AP_folds1_8':float(ap_oof_final),
           'precision_at_recall0.8_fold9':p80,'recall_oriented_fold9':{'thr':thr90,'TP':tp,'FP':fp,'FN':fn,'TN':tn},'xgb_params':{**CFG}})

### Block 7a.4b: Ensemble-value preview (SECONDARY) — OOF corr + quick stack with M1/M2

In [ ]:
# Ensemble-value PREVIEW (SECONDARY). Corr of M3 OOF with M1/M2 OOF + quick stack. Fold 10 untouched.
from sklearn.linear_model import LogisticRegression
mm=d8[['ecg_id','label']].copy(); mm['m3']=oof_raw; have={}
p1=os.path.join(PROCESSED,'m1_combined_oof.csv')
if os.path.exists(p1):
    o1=pd.read_csv(p1,dtype={'ecg_id':str})[['ecg_id','proba_raw']].rename(columns={'proba_raw':'m1'}); mm=mm.merge(o1,on='ecg_id',how='left'); have['m1']=True
p2=os.path.join(ROOT,'models','M2_statistical','m2_oof_folds1_8.npy')
if os.path.exists(p2):
    a2=np.load(p2)
    if len(a2)==len(mm): mm['m2']=a2; have['m2']=True
    else: print(f"[skip M2] len {len(a2)} != {len(mm)}")
msk=mm['m3'].notna()
print("Spearman corr of OOF scores (lower = more orthogonal):")
for k in have:
    s=mm[msk & mm[k].notna()]; print(f"  M3 vs {k.upper()}: rho = {s[['m3',k]].corr(method='spearman').iloc[0,1]:.3f}")
for base in ('m2','m1'):
    if base in have:
        s=mm[msk & mm[base].notna()].copy(); X=s[[base,'m3']].fillna(s[[base,'m3']].mean())
        apb=average_precision_score(s.label,s[base]); st=LogisticRegression(max_iter=1000).fit(X,s.label)
        aps_=average_precision_score(s.label,st.predict_proba(X)[:,1])
        print(f"  OOF AP {base.upper()} {apb:.3f} -> {base.upper()}+M3 {aps_:.3f} (gain {aps_-apb:+.3f})")
print("  NB: secondary diagnostic; primary = best standalone M3. Fold 10 untouched.")

### Block 7a.5: Freeze M3 (canonical m3_combined_* artifacts)

In [ ]:
# Freeze M3 (canonical m3_combined_* artifacts). Fold 10 never touched.
import joblib
from sklearn.calibration import CalibratedClassifierCV
from datetime import datetime
oof_cal=platt.predict_proba(oof_raw.reshape(-1,1))[:,1]
final_cal=CalibratedClassifierCV(make_final(),method='sigmoid',cv=3).fit(X8,y8)
joblib.dump(xgb_raw,os.path.join(MODELS,f'm3_{TAG}_model_raw.joblib'))
joblib.dump(final_cal,os.path.join(MODELS,f'm3_{TAG}_model.joblib'))
joblib.dump(platt,os.path.join(MODELS,f'm3_{TAG}_platt.joblib'))
oof_df=d8[['ecg_id','patient_id','fold','source','label']].copy(); oof_df['proba_raw']=oof_raw; oof_df['proba_cal']=oof_cal
oof_df.to_csv(os.path.join(PROCESSED,f'm3_{TAG}_oof.csv'),index=False)
pd.Series(FEATURES_K,name='feature').to_csv(os.path.join(MODELS,f'm3_{TAG}_features.csv'),index=False)
config={'model':f'M3_wavelet_{TAG}','version':'1.0','pool':POOL,'train':'PTB-XL + Ningbo, folds 1-8',
 'representation':'wavelet time-frequency LOCALIZATION (SWT db4+sym4+coif3 + WPT db4 L6 + DWT db4) + delineation-free '
                  'QRS-localized families (QRS-mask, QRS-onset rising-edge, polarity); energy/entropy dropped',
 'K':int(K),'cfg':CFGNAME,'depth':int(best.depth),'features':FEATURES_K,'xgb_params':{**CFG,'scale_pos_weight':float(spw8)},
 'calibration':'Platt on OOF native folds (anti-shuffle)',
 'selection':'gate |d|>0.3 & p_FDR<0.05 (BH) & IC95 & cross-dataset; Spearman>0.95 dedup; localization+QRS pool; '
             'train-gap dropped (AP_train saturated); MAX OOF AP, depth open (2/3/4) + parsimony tiebreak among true OOF-ties (smallest K); '
             'inter-fold OOF AP variance + fold9 guards. NOTE: differs from M1/M2 (1-SE) on purpose - see header. Final test = fold10 (ensemble).',
 'qrs_ablation':qrs_abl,'metrics_standard_fold9':M3_STD,'multiseed':MULTISEED,
 'fold10':'UNTOUCHED','date_frozen':datetime.now().strftime('%Y-%m-%d %H:%M')}
json.dump(config,open(os.path.join(MODELS,f'm3_{TAG}_config.json'),'w',encoding='utf-8'),ensure_ascii=False,indent=2)
print(f"M3 {TAG} FROZEN | POOL={POOL} | K={K} depth{int(best.depth)} | OOF AP {ap_oof_final:.3f} | "
      f"fold9 AP {M3_STD['AP']:.3f} | AUC {M3_STD['AUC']:.3f} | F1 {M3_STD['confusion_at_threshold']['f1']:.3f}")
print("  Fold 10 never touched. M3 combined (07a) done.")

### Results & interpretation

**Key finding:** the most discriminative features are the **delta-wave polarity at QRS onset** in lateral leads (`swtdb4_D6_I/AVL_qrspolsigned`), with the sign **flipping** between lateral (I, AVL) and inferior/AVR leads — consistent with the delta-wave axis, captured **without delineation**. QRS-onset localization (`qrson2mid`) and the low-frequency magnitude (`D6/A6_p95` in AVL/I) also rank high.

**Performance (OOF scale, comparable across detectors):** M3 reaches the strongest standalone OOF AP of the suite, well above M1 (OOF 0.198) and M2 (OOF 0.299), and is highly orthogonal to both → a strong, complementary ensemble member.

**Limitations:** at ~0.2% prevalence, precision at high recall is intrinsically low (screening detector); the score is bimodal (M3 confidently catches part of the WPW, near-blind to the rest) → the ensemble (M1–M7) covers the remainder. The OOF AP is selection-optimistic; the honest external estimate is the untouched **fold 10**.